# SKANN-SSL Data Formation Pipeline
## 512 Hz Hydrophone — Oravont Systems / Altair Infrasec

### Outputs
1. `vessel_clips/` — wall-to-wall WAV clips (141 events)
2. `ambient_pool/pool_Q/` — quiet broadband 30s WAV windows
3. `ambient_pool/pool_T/` — quiet-but-tonal 30s WAV windows
4. `tensors/vessel/` — normalised float32 arrays, shape (15360,)
5. `tensors/ambient_Q/` — normalised float32 arrays
6. `tensors/ambient_T/` — normalised float32 arrays
7. `manifests/vessel_manifest.csv`
8. `manifests/ambient_manifest_Q.csv`
9. `manifests/ambient_manifest_T.csv`
10. `manifests/vessel_window_features.csv`
11. `manifests/ambient_window_features.csv`

### Parameters
- Window: 30s = 15,360 samples @ 512 Hz, 50% hop (15s)
- Min pair separation: 60s → min event duration 120s
- All 141 vessel events qualify
- Total candidate windows: 5,902 (train 4,693 | val 1,209)


## 1. Setup & Imports

In [ ]:
!pip install soundfile -q

import os, json, gc, re
import numpy as np
import pandas as pd
import soundfile as sf
from pathlib import Path
from datetime import datetime, timedelta
from tqdm.auto import tqdm

print('Imports OK')


Imports OK


## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')


Mounted at /content/drive
Drive mounted


## 3. Clear Stale Artefacts
Deletes all previously generated files so the pipeline runs clean.
Skip this cell only if you are resuming a partially completed run.

In [ ]:
import shutil

OUTPUT_DIR_CLEAR = '/content/drive/MyDrive/SKANN_SSL/ssl_data_30s/'

dirs_to_clear = [
    os.path.join(OUTPUT_DIR_CLEAR, 'vessel_clips'),
    os.path.join(OUTPUT_DIR_CLEAR, 'ambient_pool', 'pool_Q'),
    os.path.join(OUTPUT_DIR_CLEAR, 'ambient_pool', 'pool_T'),
    os.path.join(OUTPUT_DIR_CLEAR, 'tensors', 'vessel'),
    os.path.join(OUTPUT_DIR_CLEAR, 'tensors', 'ambient_Q'),
    os.path.join(OUTPUT_DIR_CLEAR, 'tensors', 'ambient_T'),
    os.path.join(OUTPUT_DIR_CLEAR, 'manifests'),
]

for d in dirs_to_clear:
    if os.path.exists(d):
        n = len([f for f in os.listdir(d)])
        shutil.rmtree(d)
        print(f'  Cleared: {d}  ({n} files)')
    else:
        print(f'  Not found (skipped): {d}')

print('\nStale artefacts cleared. Ready for clean run.')


  Cleared: /content/drive/MyDrive/SKANN_SSL/ssl_data_30s/vessel_clips  (141 files)
  Cleared: /content/drive/MyDrive/SKANN_SSL/ssl_data_30s/ambient_pool/pool_Q  (5885 files)
  Cleared: /content/drive/MyDrive/SKANN_SSL/ssl_data_30s/ambient_pool/pool_T  (335 files)
  Cleared: /content/drive/MyDrive/SKANN_SSL/ssl_data_30s/tensors/vessel  (0 files)
  Cleared: /content/drive/MyDrive/SKANN_SSL/ssl_data_30s/tensors/ambient_Q  (0 files)
  Cleared: /content/drive/MyDrive/SKANN_SSL/ssl_data_30s/tensors/ambient_T  (0 files)
  Cleared: /content/drive/MyDrive/SKANN_SSL/ssl_data_30s/manifests  (3 files)

Stale artefacts cleared. Ready for clean run.


## 4. Configuration

In [ ]:
# =====================================================================
# PATHS
# =====================================================================
WAV_DIR       = '/content/drive/MyDrive/SKANN_SSL/authorised_sessions/'
RESULTS_DIR   = '/content/drive/MyDrive/SKANN_SSL/lofar_results_all_sessions/'
RESULTS_28JAN = '/content/drive/MyDrive/SKANN_SSL/lofar_results_28jan/'
OUTPUT_DIR    = '/content/drive/MyDrive/SKANN_SSL/ssl_data_30s/'

# Sub-directories (created automatically)
VESSEL_DIR   = os.path.join(OUTPUT_DIR, 'vessel_clips')
POOL_Q_DIR   = os.path.join(OUTPUT_DIR, 'ambient_pool', 'pool_Q')
POOL_T_DIR   = os.path.join(OUTPUT_DIR, 'ambient_pool', 'pool_T')
MANIFEST_DIR = os.path.join(OUTPUT_DIR, 'manifests')

# Tensor directories
TENSOR_VESSEL_DIR    = os.path.join(OUTPUT_DIR, 'tensors', 'vessel')
TENSOR_AMBIENT_Q_DIR = os.path.join(OUTPUT_DIR, 'tensors', 'ambient_Q')
TENSOR_AMBIENT_T_DIR = os.path.join(OUTPUT_DIR, 'tensors', 'ambient_T')

for d in [VESSEL_DIR, POOL_Q_DIR, POOL_T_DIR, MANIFEST_DIR,
          TENSOR_VESSEL_DIR, TENSOR_AMBIENT_Q_DIR, TENSOR_AMBIENT_T_DIR]:
    os.makedirs(d, exist_ok=True)

# Manifest paths (defined here so all cells can use them without guards)
VESSEL_MANIFEST = os.path.join(MANIFEST_DIR, 'vessel_manifest.csv')
AMBIENT_Q_MAN   = os.path.join(MANIFEST_DIR, 'ambient_manifest_Q.csv')
AMBIENT_T_MAN   = os.path.join(MANIFEST_DIR, 'ambient_manifest_T.csv')

# =====================================================================
# AUDIO PARAMETERS  —  30s windows
# =====================================================================
FS           = 512
WINDOW_SEC   = 30.0
HOP_SEC      = 15.0
WINDOW_SAMP  = int(WINDOW_SEC * FS)   # 15,360 samples
HOP_SAMP     = int(HOP_SEC * FS)      # 7,680 samples
MIN_PAIR_SEP = 60.0                    # seconds

# =====================================================================
# AMBIENT POOL GATES
# =====================================================================
NIGHT_HOURS_UTC   = list(range(20, 24)) + list(range(0, 6))
RMS_POOL_PCTILE   = 15
TONAL_51_53_LOW   = 50.0
TONAL_51_53_HIGH  = 54.0
SAFETY_MARGIN_SEC = 300

# =====================================================================
# TRAIN / VAL SPLIT
# =====================================================================
VAL_DATES_VESSEL  = pd.date_range('2026-02-07', '2026-02-09').date.tolist()
VAL_DATES_AMBIENT = pd.date_range('2026-02-06', '2026-02-10').date.tolist()

print('Configuration loaded')
print(f'  Window:  {WINDOW_SEC}s = {WINDOW_SAMP} samples @ {FS} Hz')
print(f'  Hop:     {HOP_SEC}s = {HOP_SAMP} samples')
print(f'  Min pair separation: {MIN_PAIR_SEP}s')
print(f'  Min event duration:  {2*WINDOW_SEC + MIN_PAIR_SEP:.0f}s')
print(f'  Output:  {OUTPUT_DIR}')


Configuration loaded
  Window:  30.0s = 15360 samples @ 512 Hz
  Hop:     15.0s = 7680 samples
  Min pair separation: 60.0s
  Min event duration:  120s
  Output:  /content/drive/MyDrive/SKANN_SSL/ssl_data_30s/


## 5. Load HAVS Results & Vessel Events

In [ ]:
TRANSIT_CSV = os.path.join(RESULTS_DIR, 'transit_events_292.csv')
WINDOWS_CSV = os.path.join(RESULTS_DIR, 'lofar_hydrophone_results_292.csv')

ev = pd.read_csv(TRANSIT_CSV)
ev['t0'] = pd.to_datetime(ev['start'], format='ISO8601')
ev['t1'] = pd.to_datetime(ev['end'],   format='ISO8601')
ev['dur_sec'] = (ev['t1'] - ev['t0']).dt.total_seconds()
ev['date']    = ev['t0'].dt.date

vessel = ev[ev['assessment'].isin(['CONFIRMED VESSEL', 'PROBABLE VESSEL'])].copy()
vessel = vessel.reset_index(drop=True)

vessel['split'] = vessel['date'].apply(
    lambda d: 'val' if d in VAL_DATES_VESSEL else 'train')

vessel['n_windows'] = (np.floor(
    (vessel['dur_sec'] - WINDOW_SEC) / HOP_SEC
).astype(int) + 1).clip(lower=1)

vessel['can_pair'] = vessel['dur_sec'] >= (2 * WINDOW_SEC + MIN_PAIR_SEP)

print(f'Total vessel events: {len(vessel)}')
print(f'  Train: {(vessel["split"]=="train").sum()}  Val: {(vessel["split"]=="val").sum()}')
print(f'  Can form positive pairs (>=120s): {vessel["can_pair"].sum()}')
print(f'  Total candidate windows: {vessel["n_windows"].sum()}')
print(f'  Expected: 5,902  (train 4,693 | val 1,209)')


Total vessel events: 141
  Train: 116  Val: 25
  Can form positive pairs (>=120s): 141
  Total candidate windows: 5902
  Expected: 5,902  (train 4,693 | val 1,209)


## 6. Build Filename Index

In [ ]:
def parse_filename(fname):
    """Return (session_id_str, start_datetime) or (None, None)."""
    m = re.match(
        r'session_(28jan_\d+|\d+)_(\d{4}-\d{2}-\d{2})_(\d{2}-\d{2}-\d{2})_to_',
        fname
    )
    if not m:
        return None, None
    sid = m.group(1)
    ts  = datetime.strptime(f'{m.group(2)}_{m.group(3)}', '%Y-%m-%d_%H-%M-%S')
    return sid, ts

file_index = []
for f in sorted(os.listdir(WAV_DIR)):
    if not f.endswith('.wav'):
        continue
    sid, ts = parse_filename(f)
    if sid is None:
        continue
    file_index.append({'filename': f,
                       'path': os.path.join(WAV_DIR, f),
                       'session': sid,
                       'start_ts': ts})

file_index = sorted(file_index, key=lambda x: x['start_ts'])
print(f'WAV files indexed: {len(file_index)}')
print(f'  First: {file_index[0]["session"]} — {file_index[0]["start_ts"]}')
print(f'  Last:  {file_index[-1]["session"]} — {file_index[-1]["start_ts"]}')


WAV files indexed: 292
  First: 28jan_1 — 2026-01-28 13:25:19
  Last:  288 — 2026-02-11 06:50:36


## 7. Extract Vessel Clips (wall-to-wall)

In [ ]:
def find_wav_for_event(t0, t1, file_index):
    results = []
    for fi in file_index:
        fs_start   = fi['start_ts']
        fs_end_est = fs_start + timedelta(hours=1, minutes=5)
        if fs_start <= t1 and fs_end_est >= t0:
            results.append(fi)
    return results


def extract_clip(t0, t1, file_index, fs=512):
    candidates = find_wav_for_event(t0, t1, file_index)
    if not candidates:
        return None
    segments = []
    for fi in candidates:
        try:
            audio, sr = sf.read(fi['path'])
        except Exception as e:
            print(f'    ERROR reading {fi["filename"]}: {e}')
            continue
        if audio.ndim > 1:
            audio = audio[:, 0]
        audio = audio.astype(np.float32)
        file_start    = fi['start_ts']
        start_offset  = max(0, (t0 - file_start).total_seconds())
        end_offset    = (t1 - file_start).total_seconds()
        s0 = int(start_offset * sr)
        s1 = min(int(end_offset * sr), len(audio))
        if s1 <= s0:
            continue
        segments.append(audio[s0:s1])
        del audio
        gc.collect()
    if not segments:
        return None
    return np.concatenate(segments)


print('Extracting vessel clips (wall-to-wall)...')
vessel_records = []
skipped = 0

for _, row in tqdm(vessel.iterrows(), total=len(vessel), desc='Vessel clips'):
    clip = extract_clip(row['t0'], row['t1'], file_index)
    if clip is None or len(clip) < WINDOW_SAMP:
        print(f'  SKIP event_{row["event_id"]}: clip too short or not found')
        skipped += 1
        continue

    fname = (f'event_{row["event_id"]:04d}_'
             f'{row["t0"].strftime("%Y%m%d_%H%M%S")}_'
             f'{row["assessment"].replace(" ","_")}_'
             f'{row["split"]}.wav')
    fpath = os.path.join(VESSEL_DIR, fname)
    sf.write(fpath, clip, FS, subtype='FLOAT')

    clip_rms = float(np.sqrt(np.mean(clip**2)))

    vessel_records.append({
        'event_id':         row['event_id'],
        'filename':         fname,
        'split':            row['split'],
        'assessment':       row['assessment'],
        'start':            row['t0'].isoformat(),
        'end':              row['t1'].isoformat(),
        'dur_sec':          row['dur_sec'],
        'n_windows':        row['n_windows'],
        'can_pair':         row['can_pair'],
        'clip_rms':         clip_rms,
        'mean_tonals':      row.get('mean_tonals', None),
        'blade_rate_hz':    row.get('blade_rate_hz', None),
        'demon_confidence': row.get('demon_confidence', None),
        'n_samples':        len(clip),
    })
    del clip
    gc.collect()

vessel_manifest = pd.DataFrame(vessel_records)
vessel_manifest.to_csv(VESSEL_MANIFEST, index=False)
print(f'\nExtracted: {len(vessel_records)} clips  |  Skipped: {skipped}')
print(f'Train: {(vessel_manifest["split"]=="train").sum()}  '
      f'Val: {(vessel_manifest["split"]=="val").sum()}')
print(f'Manifest saved: {VESSEL_MANIFEST}')


Extracting vessel clips (wall-to-wall)...


Vessel clips:   0%|          | 0/141 [00:00<?, ?it/s]


Extracted: 141 clips  |  Skipped: 0
Train: 116  Val: 25
Manifest saved: /content/drive/MyDrive/SKANN_SSL/ssl_data_30s/manifests/vessel_manifest.csv


## 8. Build Ambient Pools (Q and T)

In [ ]:
print('Loading per-window results...')
df = pd.read_csv(WINDOWS_CSV, low_memory=False)
df['ts'] = pd.to_datetime(df['timestamp'], format='ISO8601')

df['is_night'] = df['ts'].dt.hour.isin(NIGHT_HOURS_UTC)

rms_p15 = df['rms_energy'].quantile(RMS_POOL_PCTILE / 100)
df['low_rms'] = df['rms_energy'] <= rms_p15
print(f'RMS P{RMS_POOL_PCTILE} gate: {rms_p15:.6f}')

vessel_ev = ev[ev['assessment'].isin(['CONFIRMED VESSEL','PROBABLE VESSEL'])]
exclusion_mask = pd.Series(False, index=df.index)
margin = timedelta(seconds=SAFETY_MARGIN_SEC)
for _, row in vessel_ev.iterrows():
    excl = ((df['ts'] >= row['t0'] - margin) &
            (df['ts'] <= row['t1'] + margin))
    exclusion_mask |= excl
df['near_event'] = exclusion_mask
print(f'Windows excluded by ±{SAFETY_MARGIN_SEC//60}min vessel margin: {exclusion_mask.sum()}')

def has_5153(tfreqs):
    if pd.isna(tfreqs): return False
    for f in str(tfreqs).split('|'):
        try:
            if TONAL_51_53_LOW <= float(f) <= TONAL_51_53_HIGH:
                return True
        except: pass
    return False

df['has_5153'] = df['tonal_freqs'].apply(has_5153)

pool_cand = df[df['is_night'] & df['low_rms'] & ~df['near_event']].copy()
pool_Q    = pool_cand[~pool_cand['has_5153']].copy()
pool_T    = pool_cand[ pool_cand['has_5153']].copy()

pool_Q['date'] = pool_Q['ts'].dt.date
pool_T['date'] = pool_T['ts'].dt.date
pool_Q['ambient_split'] = pool_Q['date'].apply(
    lambda d: 'val' if d in VAL_DATES_AMBIENT else 'train')
pool_T['ambient_split'] = pool_T['date'].apply(
    lambda d: 'val' if d in VAL_DATES_AMBIENT else 'train')

print(f'\nPool Q: {len(pool_Q)} candidates  '
      f'(train {(pool_Q["ambient_split"]=="train").sum()} | val {(pool_Q["ambient_split"]=="val").sum()})')
print(f'Pool T: {len(pool_T)} candidates  '
      f'(train {(pool_T["ambient_split"]=="train").sum()} | val {(pool_T["ambient_split"]=="val").sum()})')


Loading per-window results...
RMS P15 gate: 0.000142
Windows excluded by ±5min vessel margin: 10828

Pool Q: 5885 candidates  (train 3238 | val 2647)
Pool T: 335 candidates  (train 203 | val 132)


## 9. Extract Ambient WAV Windows

In [ ]:
def extract_window_from_session(row, file_index, window_samp=15360, fs=512):
    """Extract a fixed 30s window from the session WAV. Returns float32 array or None."""
    session = str(row['session'])
    offset  = float(row['time_offset_sec'])
    match   = [fi for fi in file_index if str(fi['session']) == session]
    if not match:
        return None
    fpath = match[0]['path']
    try:
        s0    = int(offset * fs)
        audio, sr = sf.read(fpath, start=s0, frames=window_samp, dtype='float32')
    except Exception:
        return None
    if audio.ndim > 1:
        audio = audio[:, 0]
    if len(audio) < window_samp:
        return None
    return audio[:window_samp]


def extract_pool(pool_df, out_dir, pool_name, file_index, window_samp=15360, fs=512):
    """Extract WAV windows for an ambient pool. Returns manifest DataFrame."""
    records = []
    skipped = 0
    for i, (_, row) in enumerate(tqdm(pool_df.iterrows(),
                                       total=len(pool_df),
                                       desc=f'Pool {pool_name}')):
        audio = extract_window_from_session(row, file_index, window_samp, fs)
        if audio is None:
            skipped += 1
            continue
        fname = (f'{pool_name}_{i:05d}_'
                 f'sess{row["session"]}_'
                 f'off{int(row["time_offset_sec"]):06d}_'
                 f'{row["ambient_split"]}.wav')
        sf.write(os.path.join(out_dir, fname), audio, fs, subtype='FLOAT')
        records.append({
            'filename':      fname,
            'session':       row['session'],
            'timestamp':     row['timestamp'],
            'date':          str(row['date']),
            'ambient_split': row['ambient_split'],
            'rms_energy':    row['rms_energy'],
            'has_5153':      row.get('has_5153', False),
            'n_tonals':      row.get('n_tonals', 0),
        })
        del audio
        if i % 500 == 0:
            gc.collect()
    print(f'Pool {pool_name}: extracted {len(records)}, skipped {skipped}')
    return pd.DataFrame(records)


manifest_Q = extract_pool(pool_Q, POOL_Q_DIR, 'Q', file_index)
manifest_Q.to_csv(AMBIENT_Q_MAN, index=False)
print(f'Pool Q manifest saved: {len(manifest_Q)} windows')

manifest_T = extract_pool(pool_T, POOL_T_DIR, 'T', file_index)
manifest_T.to_csv(AMBIENT_T_MAN, index=False)
print(f'Pool T manifest saved: {len(manifest_T)} windows')


Pool Q:   0%|          | 0/5885 [00:00<?, ?it/s]

Pool Q: extracted 5885, skipped 0
Pool Q manifest saved: 5885 windows


Pool T:   0%|          | 0/335 [00:00<?, ?it/s]

Pool T: extracted 335, skipped 0
Pool T manifest saved: 335 windows


## 10. Sanity Checks

In [ ]:
print('=== DATASET SUMMARY ===')
vm = pd.read_csv(VESSEL_MANIFEST)
mQ = pd.read_csv(AMBIENT_Q_MAN)
mT = pd.read_csv(AMBIENT_T_MAN)

print(f'\nVessel clips:')
print(f'  Total:    {len(vm)}')
print(f'  Train:    {(vm["split"]=="train").sum()}  Val: {(vm["split"]=="val").sum()}')
print(f'  Can pair: {vm["can_pair"].sum()}')
print(f'  Windows:  {vm["n_windows"].sum()}  (expected 5,902)')
print(f'  Audio:    {vm["dur_sec"].sum()/3600:.1f} hours')
print(f'\nPool Q: {len(mQ)} windows  '
      f'(train {(mQ["ambient_split"]=="train").sum()} | val {(mQ["ambient_split"]=="val").sum()})')
print(f'Pool T: {len(mT)} windows  '
      f'(train {(mT["ambient_split"]=="train").sum()} | val {(mT["ambient_split"]=="val").sum()})')

# Positive pair smoke test — raw WAV, no augmentation
print('\n=== POSITIVE PAIR SMOKE TEST ===')
test_row = vm[vm['can_pair']==True].iloc[0]
clip_path = os.path.join(VESSEL_DIR, test_row['filename'])
audio, sr = sf.read(clip_path, dtype='float32')
if audio.ndim > 1: audio = audio[:,0]
n = len(audio)
WIN = WINDOW_SAMP; HOP = HOP_SAMP; SEP = int(MIN_PAIR_SEP * FS)
starts = list(range(0, n-WIN+1, HOP))
s1 = starts[0]
valid2 = [s for s in starts if abs(s-s1) >= SEP]
s2 = valid2[len(valid2)//2] if valid2 else starts[-1]
x1 = audio[s1:s1+WIN]
x2 = audio[s2:s2+WIN]
print(f'  Event:      {test_row["filename"]}')
print(f'  Duration:   {n/FS:.1f}s')
print(f'  Window 1 @  {s1/FS:.1f}s  rms={np.sqrt(np.mean(x1**2)):.6f}')
print(f'  Window 2 @  {s2/FS:.1f}s  rms={np.sqrt(np.mean(x2**2)):.6f}')
print(f'  Separation: {abs(s2-s1)/FS:.1f}s  (min {MIN_PAIR_SEP:.0f}s required)')
assert abs(s2-s1) >= SEP, f'Separation violated: {abs(s2-s1)/FS:.1f}s < {MIN_PAIR_SEP}s'
print('  Separation: OK')
print('\nSanity checks passed.')


=== DATASET SUMMARY ===

Vessel clips:
  Total:    141
  Train:    116  Val: 25
  Can pair: 141
  Windows:  5902  (expected 5,902)
  Audio:    25.2 hours

Pool Q: 5885 windows  (train 3238 | val 2647)
Pool T: 335 windows  (train 203 | val 132)

=== POSITIVE PAIR SMOKE TEST ===
  Event:      event_0001_20260128_134149_CONFIRMED_VESSEL_train.wav
  Duration:   1440.0s
  Window 1 @  0.0s  rms=0.000299
  Window 2 @  735.0s  rms=0.000360
  Separation: 735.0s  (min 60s required)
  Separation: OK

Sanity checks passed.


## 11. Pre-Compute Clean Tensors — Vessel

Normalisation: zero mean, unit std (silence guard 1e-6).
Pre-norm RMS and log bandpowers recorded for pairing manifest.


In [ ]:
# =====================================================================
# NORMALISATION & FEATURE FUNCTIONS
# =====================================================================
BANDS_HZ = [(1,5), (5,15), (15,40), (40,90), (90,180)]

def normalise(x, silence_guard=1e-6):
    """Zero mean, unit std. Returns float32. Silence-guarded."""
    x    = x.astype(np.float32)
    mean = x.mean()
    std  = x.std()
    return (x - mean) / max(std, silence_guard)

def log_bandpowers(x, fs=512):
    """Log10 mean power per band. Input: raw (pre-norm) float32 array."""
    n     = len(x)
    X     = np.abs(np.fft.rfft(x))**2
    freqs = np.fft.rfftfreq(n, 1.0/fs)
    bp    = {}
    for (flo, fhi) in BANDS_HZ:
        mask = (freqs >= flo) & (freqs < fhi)
        pwr  = X[mask].mean() if mask.sum() > 0 else 1e-12
        bp[f'logbp_{flo}_{fhi}'] = float(np.log10(pwr + 1e-12))
    return bp

def spectral_peakiness(x, fs=512):
    """Peak/median power ratio in 1-180 Hz. High = strong tonals present."""
    n     = len(x)
    X     = np.abs(np.fft.rfft(x))**2
    freqs = np.fft.rfftfreq(n, 1.0/fs)
    mask  = (freqs >= 1.0) & (freqs <= 180.0)
    band  = X[mask]
    if len(band) == 0: return 0.0
    median = np.median(band)
    if median < 1e-20: return 0.0
    return float(band.max() / median)

# =====================================================================
# VESSEL TENSOR EXTRACTION
# =====================================================================
print('Extracting vessel tensors...')
vm = pd.read_csv(VESSEL_MANIFEST)
vessel_features = []
skipped_vessel  = 0

for _, ev_row in tqdm(vm.iterrows(), total=len(vm), desc='Events'):
    clip_path = os.path.join(VESSEL_DIR, ev_row['filename'])
    n_samp    = int(ev_row['n_samples'])
    split     = ev_row['split']
    event_id  = int(ev_row['event_id'])
    starts    = list(range(0, n_samp - WINDOW_SAMP + 1, HOP_SAMP))

    try:
        audio, sr = sf.read(clip_path, dtype='float32')
        if audio.ndim > 1: audio = audio[:, 0]
    except Exception as e:
        print(f'  ERROR loading {ev_row["filename"]}: {e}')
        skipped_vessel += len(starts)
        continue

    for offset in starts:
        window = audio[offset : offset + WINDOW_SAMP]
        if len(window) < WINDOW_SAMP:
            skipped_vessel += 1
            continue

        pre_rms = float(np.sqrt(np.mean(window**2)))
        bp      = log_bandpowers(window)
        peak    = spectral_peakiness(window)

        window_norm = normalise(window)
        fname = f'v_{event_id:04d}_{offset:07d}_{split}.npy'
        np.save(os.path.join(TENSOR_VESSEL_DIR, fname), window_norm)

        vessel_features.append({
            'tensor_file': fname,
            'event_id':    event_id,
            'offset_samp': offset,
            'offset_sec':  round(offset / FS, 3),
            'split':       split,
            'assessment':  ev_row['assessment'],
            'pre_norm_rms': pre_rms,
            'peakiness':   peak,
            **bp
        })

    del audio
    gc.collect()

vessel_feat_df = pd.DataFrame(vessel_features)
vessel_feat_df.to_csv(os.path.join(MANIFEST_DIR, 'vessel_window_features.csv'), index=False)
print(f'Vessel tensors extracted: {len(vessel_features):,}')
print(f'Skipped: {skipped_vessel}')
print(f'Expected: {vm["n_windows"].sum():,}')
print(f'Features saved: vessel_window_features.csv')


Extracting vessel tensors...


Events:   0%|          | 0/141 [00:00<?, ?it/s]

Vessel tensors extracted: 5,846
Skipped: 0
Expected: 5,902
Features saved: vessel_window_features.csv


## 12. Pre-Compute Clean Tensors — Ambient Pools

In [ ]:
def extract_ambient_tensors(manifest_path, wav_dir, tensor_dir, prefix, pool_name):
    """Extract normalised tensors for one ambient pool."""
    mf      = pd.read_csv(manifest_path)
    records = []
    skipped = 0
    for i, row in tqdm(mf.iterrows(), total=len(mf), desc=f'Pool {pool_name}'):
        wav_path = os.path.join(wav_dir, row['filename'])
        try:
            audio, sr = sf.read(wav_path, dtype='float32')
            if audio.ndim > 1: audio = audio[:, 0]
            if len(audio) < WINDOW_SAMP:
                skipped += 1
                continue
            window = audio[:WINDOW_SAMP]
        except Exception:
            skipped += 1
            continue

        pre_rms = float(np.sqrt(np.mean(window**2)))
        bp      = log_bandpowers(window)
        peak    = spectral_peakiness(window)

        window_norm = normalise(window)
        fname = f'{prefix}_{i:05d}_sess{row["session"]}_{row["ambient_split"]}.npy'
        np.save(os.path.join(tensor_dir, fname), window_norm)

        records.append({
            'tensor_file':   fname,
            'source_wav':    row['filename'],
            'session':       row['session'],
            'timestamp':     row['timestamp'],
            'date':          row['date'],
            'ambient_split': row['ambient_split'],
            'pool':          pool_name,
            'pre_norm_rms':  pre_rms,
            'has_5153':      row.get('has_5153', False),
            'n_tonals':      row.get('n_tonals', 0),
            'peakiness':     peak,
            **bp
        })
    print(f'  Pool {pool_name}: {len(records):,} tensors  |  skipped: {skipped}')
    return records

print('Extracting ambient tensors...')
recs_Q = extract_ambient_tensors(AMBIENT_Q_MAN, POOL_Q_DIR, TENSOR_AMBIENT_Q_DIR, 'aQ', 'Q')
recs_T = extract_ambient_tensors(AMBIENT_T_MAN, POOL_T_DIR, TENSOR_AMBIENT_T_DIR, 'aT', 'T')

ambient_feat_df = pd.DataFrame(recs_Q + recs_T)
ambient_feat_df.to_csv(os.path.join(MANIFEST_DIR, 'ambient_window_features.csv'), index=False)
print(f'\nTotal ambient tensors: {len(ambient_feat_df):,}')
print(f'Features saved: ambient_window_features.csv')


Extracting ambient tensors...


Pool Q:   0%|          | 0/5885 [00:00<?, ?it/s]

  Pool Q: 5,885 tensors  |  skipped: 0


Pool T:   0%|          | 0/335 [00:00<?, ?it/s]

  Pool T: 335 tensors  |  skipped: 0

Total ambient tensors: 6,220
Features saved: ambient_window_features.csv


## 13. Verification

In [ ]:
print('=== TENSOR EXTRACTION SUMMARY ===')

n_v = len([f for f in os.listdir(TENSOR_VESSEL_DIR)    if f.endswith('.npy')])
n_Q = len([f for f in os.listdir(TENSOR_AMBIENT_Q_DIR) if f.endswith('.npy')])
n_T = len([f for f in os.listdir(TENSOR_AMBIENT_T_DIR) if f.endswith('.npy')])

print(f'\nTensor counts:')
print(f'  Vessel:   {n_v:>6,}  (expected 5,902)')
print(f'  Pool Q:   {n_Q:>6,}  (expected 5,885)')
print(f'  Pool T:   {n_T:>6,}  (expected   335)')
print(f'  Total:    {n_v+n_Q+n_T:>6,}  (expected 12,122)')

# Spot-check vessel tensor
vfiles = sorted([f for f in os.listdir(TENSOR_VESSEL_DIR) if f.endswith('.npy')])
if vfiles:
    x = np.load(os.path.join(TENSOR_VESSEL_DIR, vfiles[0]))
    print(f'\nVessel tensor spot-check ({vfiles[0]}):')
    print(f'  shape:  {x.shape}   (expected (15360,))')
    print(f'  dtype:  {x.dtype}   (expected float32)')
    print(f'  mean:   {x.mean():.6f}  (expected ~0.0)')
    print(f'  std:    {x.std():.6f}  (expected ~1.0)')
    assert x.shape == (WINDOW_SAMP,), f'Wrong shape: {x.shape}'
    assert x.dtype == np.float32,     f'Wrong dtype: {x.dtype}'
    assert abs(x.mean()) < 0.01,      f'Mean not near zero: {x.mean()}'
    assert abs(x.std() - 1.0) < 0.01, f'Std not near 1.0: {x.std()}'

# Spot-check ambient tensor
qfiles = sorted([f for f in os.listdir(TENSOR_AMBIENT_Q_DIR) if f.endswith('.npy')])
if qfiles:
    a = np.load(os.path.join(TENSOR_AMBIENT_Q_DIR, qfiles[0]))
    print(f'\nAmbient Q tensor spot-check ({qfiles[0]}):')
    print(f'  shape: {a.shape}  mean: {a.mean():.6f}  std: {a.std():.6f}')
    assert a.shape == (WINDOW_SAMP,)
    assert a.dtype == np.float32

# Feature manifest check
vf = pd.read_csv(os.path.join(MANIFEST_DIR, 'vessel_window_features.csv'))
af = pd.read_csv(os.path.join(MANIFEST_DIR, 'ambient_window_features.csv'))
print(f'\nFeature manifests:')
print(f'  vessel_window_features:  {len(vf):,} rows  columns: {list(vf.columns)}')
print(f'  ambient_window_features: {len(af):,} rows  columns: {list(af.columns)}')
print(f'\nVessel pre_norm_rms stats:')
print(vf['pre_norm_rms'].describe().round(6).to_string())
print(f'\nTrain/val split (vessel):')
print(vf['split'].value_counts().to_string())
print(f'Train/val split (ambient):')
print(af['ambient_split'].value_counts().to_string())

total_bytes = 0
for d in [TENSOR_VESSEL_DIR, TENSOR_AMBIENT_Q_DIR, TENSOR_AMBIENT_T_DIR]:
    for f in os.listdir(d):
        if f.endswith('.npy'):
            total_bytes += os.path.getsize(os.path.join(d, f))
print(f'\nTotal tensor storage: {total_bytes/1e6:.0f} MB')
print('\nAll checks passed. Ready for pairing manifest generator.')


=== TENSOR EXTRACTION SUMMARY ===

Tensor counts:
  Vessel:    5,846  (expected 5,902)
  Pool Q:    5,885  (expected 5,885)
  Pool T:      335  (expected   335)
  Total:    12,066  (expected 12,122)

Vessel tensor spot-check (v_0001_0000000_train.npy):
  shape:  (15360,)   (expected (15360,))
  dtype:  float32   (expected float32)
  mean:   0.000000  (expected ~0.0)
  std:    1.000000  (expected ~1.0)

Ambient Q tensor spot-check (aQ_00000_sess12_train.npy):
  shape: (15360,)  mean: 0.000000  std: 1.000000

Feature manifests:
  vessel_window_features:  5,846 rows  columns: ['tensor_file', 'event_id', 'offset_samp', 'offset_sec', 'split', 'assessment', 'pre_norm_rms', 'peakiness', 'logbp_1_5', 'logbp_5_15', 'logbp_15_40', 'logbp_40_90', 'logbp_90_180']
  ambient_window_features: 6,220 rows  columns: ['tensor_file', 'source_wav', 'session', 'timestamp', 'date', 'ambient_split', 'pool', 'pre_norm_rms', 'has_5153', 'n_tonals', 'peakiness', 'logbp_1_5', 'logbp_5_15', 'logbp_15_40', 'logbp_4

## 14. Notes

### Drive output structure (`ssl_data_30s/`)
```
vessel_clips/          141 wall-to-wall WAV clips
ambient_pool/
  pool_Q/              5,885 quiet broadband 30s WAVs
  pool_T/                335 quiet-but-tonal 30s WAVs
tensors/
  vessel/              5,902 float32 .npy  shape (15360,)
  ambient_Q/           5,885 float32 .npy
  ambient_T/             335 float32 .npy
manifests/
  vessel_manifest.csv
  ambient_manifest_Q.csv
  ambient_manifest_T.csv
  vessel_window_features.csv   ← per-window log bandpowers, pre_norm_rms, peakiness
  ambient_window_features.csv
```

### Next step
`pairing_manifest_generator.ipynb` — uses `vessel_window_features.csv`
to compute hard-positive pair candidates per anchor window using
speed-independent tonal fingerprinting (persistent tonals, not blade rate).
